# 01 CDRMIP Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from functools import partial

from cdrmip_extremes import (
    configs, load_pipeline, io_disk, gwl, utils,
    extremes_pipeline, extremes, file_search
)


In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

2026-10-02 11:28:32,993 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f9df96db168b2d2de05f171192f341fd initialized by task ('rechunk-merge-rechunk-transfer-702b24f4e3c10ed2b97d464baf00682d', 0, 0, 0, 92, 0, 0) executed on worker tcp://127.0.0.1:34337
2026-10-02 11:31:01,091 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f9df96db168b2d2de05f171192f341fd deactivated due to stimulus 'task-finished-1790904660.8035858'
2026-10-02 11:32:25,644 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5604af041ecfdc476eb774d4a12fc14a initialized by task ('rechunk-merge-rechunk-transfer-beaf58b998a8cefaac9d0021bf0acbcc', 0, 0, 0, 74, 0, 0) executed on worker tcp://127.0.0.1:38793
2026-10-02 11:32:52,650 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5604af041ecfdc476eb774d4a12fc14a deactivated due to stimulus 'task-finished-1790904771.5217032'
2026-10-02 11:34:21,351 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b0745c3ffa247f6ddcf6202b99

In [3]:
file_search.find_CMIP_files(
    experiment_id = '1pctCO2-cdr',
    table_id = 'day',
    variable_id = 'tasmin'
)

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,version,path
0,CDRMIP,CSIRO,ACCESS-ESM1-5,1pctCO2-cdr,r1i1p1f1,day,tasmin,gn,latest,[/g/data/fs38/publications/CMIP6/CDRMIP/CSIRO/...
1,CDRMIP,CCCma,CanESM5,1pctCO2-cdr,r1i1p2f1,day,tasmin,gn,v20190429,[/g/data/if69/sc1326/esgf_cache/CMIP6/CDRMIP/C...
2,CDRMIP,MIROC,MIROC-ES2L,1pctCO2-cdr,r1i1p1f2,day,tasmin,gn,v20220506,[/g/data/if69/sc1326/esgf_cache/CMIP6/CDRMIP/M...
3,CDRMIP,CNRM-CERFACS,CNRM-ESM2-1,1pctCO2-cdr,r1i1p1f2,day,tasmin,gr,v20210121,[/g/data/if69/sc1326/esgf_cache/CMIP6/CDRMIP/C...
4,CDRMIP,MOHC,UKESM1-0-LL,1pctCO2-cdr,r1i1p1f2,day,tasmin,gn,v20200425,[/g/data/if69/sc1326/esgf_cache/CMIP6/CDRMIP/M...
5,CDRMIP,NCC,NorESM2-LM,1pctCO2-cdr,r1i1p1f1,day,tasmin,gn,v20210118,[/g/data/if69/sc1326/esgf_cache/CMIP6/CDRMIP/N...


In [4]:
config = load_pipeline.LoadConfig(
    experiments=[
        load_pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='day', variable_id='tasmin',
            anomaly_reference='piControl',grid_label=None,
            overshoot=True, crossing_gwls = [1.5], gwl_match_period = (320,340)
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [5]:
pipe = load_pipeline.LoadingPipeline(config)

In [6]:
pipe.load_raw()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'day', 'variable_id': 'tasmin', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-01 12:00:00 -> 1000-12-31 12:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-01 12:00:00 -> 2189-12-31 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-01 12:00:00 -> 2190-12-31 12:00:00
Loading MIROC-ES2L / r1i1p1f2
  1989-01-01 12:00:00 -> 2490-12-31 12:00:00
Loading NorESM2-LM / r1i1p1f1
  0141-01-01 12:00:00 -> 0399-12-31 12:00:00
Loading UKESM1-0-LL / r1i1p1f2
  1990-01-01 12:00:00 -> 2639-12-30 12:00:00


In [7]:
pipe.concatenate_to_parents()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [ACCESS-ESM1-5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 1pctCO2 r1i1p1f2
Concatenating 1pctCO2-cdr -> 1pctCO2 [CNRM-ESM2-1] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CanESM5 1pctCO2-cdr r1i1p2f1 parent
Loading CanESM5 1pctCO2 r1i1p2f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [CanESM5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for MIROC-ES2L 1pctCO2-cdr r1i1p1f2 parent
Loading MIROC-ES2L 1pctCO2 r1i1p1f2
Concatenati

In [8]:
pipe.load_piControl()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 piControl r1i1p1f2
Searching for CanESM5 1pctCO2-cdr r1i1p2f1 parent
Loading CanESM5 piControl r1i1p2f1
Searching for MIROC-ES2L 1pctCO2-cdr r1i1p1f2 parent
Loading MIROC-ES2L piControl r1i1p1f2
Searching for NorESM2-LM 1pctCO2-cdr r1i1p1f1 parent
No parent found for NorESM2-LM 1pctCO2-cdr r1i1p1f1 (parent: NorESM2-LM piControl r1i1p1f1) — skipping
Searching for UKESM1-0-LL 1pctCO2-cdr r1i1p1f2 parent
Loading UKESM1-0-LL piControl r1i1p1f2


In [9]:
pipe.regrid()

Building regridder for grid (145,) x (192,)
Building regridder for grid (128,) x (256,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (96,) x (144,)
Building regridder for grid (144,) x (192,)


In [10]:
pipe.compute_anomalies()

  No piControl for NorESM2-LM/r1i1p1f1 — skipping anomaly


In [11]:
# pipe.load(attribute='regridded_concat')

## Calculate tnn

In [12]:
tnn = utils.map_nested(extremes.calc_tnn, pipe.regridded_concat)
pi_ref = utils.map_nested(lambda ds: extremes.calc_tnn(ds).mean("year"), pipe.regridded_piControl)
tnn_anom = extremes.picontrol_anomaly(tnn, pi_ref)

In [13]:
io_disk.save_tree(
    tnn_anom,
    configs.PROC_DIR,
    'tnn',
    activity_id='CDRMIP',
    category='anomaly'
)

Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tnn_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tnn_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tnn_anomaly.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tnn_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_tnn_anomaly.nc


## Calc Tn10p

In [14]:
thresh = utils.map_nested(extremes.tn10p_threshold, pipe.regridded_piControl)
concat_340 = utils.map_nested(partial(utils.select_years, lo=0, hi=340), pipe.regridded_concat)
tn10p = utils.map_with_picontrol(extremes.calc_tn10p, concat_340, thresh)

In [15]:
io_disk.save_tree(
    tn10p,
    configs.PROC_DIR,
    'tn10p',
    activity_id='CDRMIP',
    overwrite=False
)

Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tn10p.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tn10p.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tn10p.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tn10p.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_tn10p.nc


In [21]:
tn10p

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 44MB
   Dimensions:      (lat: 90, lon: 180, year: 341)
   Coordinates:
     * lat          (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
     * lon          (lon) float64 1kB 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
     * year         (year) int64 3kB 0 1 2 3 4 5 6 ... 334 335 336 337 338 339 340
       percentiles  int64 8B 10
   Data variables:
       tn10p        (year, lat, lon) float64 44MB dask.array<chunksize=(341, 45, 45), meta=np.ndarray>}},
 'CNRM-ESM2-1': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.Dataset> Size: 44MB
   Dimensions:      (lat: 90, lon: 180, year: 340)
   Coordinates:
     * lat          (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
     * lon          (lon) float64 1kB 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
     * year         (year) int64 3kB 0 1 2 3 4 5 6 ... 333 334 335 336 337 338 339
       percentiles  int64 8B 10
   Data variables:
       tn10p    

In [9]:
model, expt, member = 'ACCESS-ESM1-5', '1pctCO2-cdr', 'r1i1p1f1'
ds = pipe.regridded_concat[model][expt][member]
pi_ds = pipe.regridded_piControl[model][member]

In [ ]:
thresh_doy = extremes.tx90p_threshold(pi_ds)

In [ ]:
tx90p = extremes.calc_tx90p(ds, thresh_doy)
tx90p

## GWL Analysis

In [16]:
crossing_years = io_disk.load_tree(configs.PROC_DIR, 'crossing_years', 'CDRMIP', as_dataarray=True)

In [17]:
tnn_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    tnn_anom, crossing_years,
)

In [18]:
tn10p_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    tn10p, crossing_years,
)

In [19]:
io_disk.save_tree(
    tnn_diffs,
    configs.PROC_DIR,
    'tnn_gwl_differences',
    activity_id='CDRMIP',
    category=None
)

Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tnn_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tnn_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tnn_gwl_differences.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn_gwl_differences/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tnn_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tnn_gwl_differences/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_tnn_gwl_differences.nc


In [20]:
io_disk.save_tree(
    tn10p_diffs,
    configs.PROC_DIR,
    'tn10p_gwl_differences',
    activity_id='CDRMIP',
    category=None
)

Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tn10p_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tn10p_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tn10p_gwl_differences.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p_gwl_differences/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tn10p_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tn10p_gwl_differences/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_tn10p_gwl_differences.nc


In [22]:
tn10p_diffs

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 132kB
   Dimensions:      (lat: 90, lon: 180, gwl: 1)
   Coordinates:
     * lat          (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
     * lon          (lon) float64 1kB 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
     * gwl          (gwl) float64 8B 1.5
       percentiles  int64 8B 10
   Data variables:
       tn10p        (gwl, lat, lon) float64 130kB dask.array<chunksize=(1, 45, 45), meta=np.ndarray>}},
 'CNRM-ESM2-1': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.Dataset> Size: 132kB
   Dimensions:      (lat: 90, lon: 180, gwl: 1)
   Coordinates:
     * lat          (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
     * lon          (lon) float64 1kB 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
     * gwl          (gwl) float64 8B 1.5
       percentiles  int64 8B 10
   Data variables:
       tn10p        (gwl, lat, lon) float64 130kB dask.array<chunksize=(1, 45, 45), meta=np.ndarray>}},
 '